# Baselines sweep — plots

Figure D.2-style grid for the 4-method comparison (CMM-logistic, PC, GES, empty) on mixed binary/continuous data.
Loads `results/baselines_20260502_0415/results.csv`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from experiments.mixed_cmm.synthetic.config import PARAM_LABELS
from src_tb.causal_recovery.visualization import plot_sweep

RESULTS = Path.cwd().parent / 'results' / 'baselines_20260502_0415' / 'results.csv'
df = pd.read_csv(RESULTS)
df.head()

In [ ]:
method_labels = {
    'cmm_logistic': 'CMM (logistic)',
    'pc_pillai':    'PC (Pillai)',
    'ges_cg':       'GES (CG)',
    'empty':        'Empty',
}
methods_order = ['cmm_logistic', 'pc_pillai', 'ges_cg', 'empty']
df_plot = df[df['method'].isin(methods_order)].copy()

## Overall graph metrics (Figure D.2 style)

In [ ]:
metric_labels = {
    'sd':  'SD',
    'sc':  'S/C',
    'shd': 'SHD',
    'fpr': 'FPR-dir',
    'tpr': 'TPR-dir',
    'f1':  'F1-dir',
}
plot_sweep(df_plot, metrics=list(metric_labels), metric_labels=metric_labels,
           param_labels=PARAM_LABELS, group_col='method', group_labels=method_labels,
           title='Causal recovery on mixed binary/continuous data')

## Edge-type breakdown

Splits F1 by edge class: mutation→mutation (`bin_bin`) vs mutation→MIC (`bin_cont`).

In [ ]:
edge_metric_labels = {
    'f1_bin_bin':       'F1 (mut→mut)',
    'f1_bin_cont':      'F1 (mut→Y)',
    'precision_bin_bin':'Prec (mut→mut)',
    'precision_bin_cont':'Prec (mut→Y)',
    'recall_bin_bin':   'Rec (mut→mut)',
    'recall_bin_cont':  'Rec (mut→Y)',
}
plot_sweep(df_plot, metrics=list(edge_metric_labels), metric_labels=edge_metric_labels,
           param_labels=PARAM_LABELS, group_col='method', group_labels=method_labels,
           title='Recovery by edge type')

## Defaults-only summary table

Mean metrics at each method's default-config rows (n_obs=10, p_graph=0.4, p_mix=0.5, n_mix=2, n_samples=164).

In [ ]:
from experiments.mixed_cmm.synthetic.config import DEFAULTS
param_to_default = {'n_mix': DEFAULTS['n_mix'], 'p_mix': DEFAULTS['p_mix'],
                    'n_samples': DEFAULTS['n_samples'], 'n_obs': DEFAULTS['n_obs'],
                    'p_graph': DEFAULTS['p_graph']}
mask = df_plot.apply(lambda r: r['value'] == param_to_default.get(r['param']), axis=1)
df_def = df_plot[mask]
summary = df_def.groupby('method')[['f1', 'tpr', 'fpr', 'shd', 'f1_bin_bin', 'f1_bin_cont']].mean().round(3)
summary.loc[methods_order]